In [ ]:
!pip install torch


[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


: 

In [ ]:
import torch
import torch.nn as nn


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\jaden\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\jaden\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\jaden\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
   

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


class MultiHeadTemporalGAT(MessagePassing):
    def __init__(self, node_dim, edge_emb_dim, num_edge_types, heads=4, dropout=0.1):
        super().__init__(aggr="add")

        # make sure sizing is right
        assert node_dim % heads == 0, "node_dim must be divisible by heads"

        # attributes
        self.node_dim = node_dim
        self.heads = heads
        self.head_dim = node_dim // heads
        self.dropout = dropout

        # edge embeddings
        self.edge_embedding = nn.Embedding(num_edge_types, edge_emb_dim)

        # projections
        self.W_h = nn.Linear(node_dim, node_dim, bias=False) # for node embeddings in message
        self.W_r = nn.Linear(edge_emb_dim, node_dim, bias=False) # for edge embeddings in message
        self.W_n = nn.Linear(node_dim, node_dim, bias=False) # for node embeddings in attention

        # attention MLP operates per head
        # expects concat of both node embeddings and the edge embedding
        # outputs a scalar, the attention score
        self.att_mlp = nn.Sequential(
            nn.Linear(2 * self.head_dim + edge_emb_dim, self.head_dim),
            nn.LeakyReLU(),
            nn.Linear(self.head_dim, 1)
        )

        # output and normalization
        self.out_proj = nn.Linear(node_dim, node_dim)
        self.norm = nn.LayerNorm(node_dim)

    def forward(self, x, edge_index, edge_type):
        # get edge embeddings
        edge_attr = self.edge_embedding(edge_type)

        # propagates messages
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)

        # serves to mix info from different heads
        out = self.out_proj(out)

        # residual and norm
        return self.norm(x + out)

    # message is called in propagate
    def message(self, x_i, x_j, edge_attr, index):
        # apply transfomation to node embeddings
        z_i = self.W_n(x_i)
        z_j = self.W_n(x_j)

        # apply message transfomation to node embedding
        m_j = self.W_h(x_j)

        # apply message transformation to edge embedding and project to node space
        e_j = self.W_r(edge_attr)

        # reshape (E, heads, head_dim)
        z_i = z_i.view(-1, self.heads, self.head_dim)
        z_j = z_j.view(-1, self.heads, self.head_dim)
        m_j = m_j.view(-1, self.heads, self.head_dim)
        e_j = e_j.view(-1, self.heads, self.head_dim)

        # expand edge_attr to heads
        edge_attr = edge_attr.unsqueeze(1).repeat(1, self.heads, 1)

        # attention per head
        att_input = torch.cat([z_i, z_j, edge_attr], dim=-1)
        e_ij = self.att_mlp(att_input).squeeze(-1)   # [E, heads]

        # apply softmax to attention scores to get attention weights
        alpha = softmax(e_ij, index)                 # normalize per node
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        # calculate messages
        msg = m_j + e_j                              # [E, heads, head_dim]

        # weight messages and return
        return alpha.unsqueeze(-1) * msg             # weighted messages

    def aggregate(self, inputs, index, ptr=None, dim_size=None):
        # inputs: [E, heads, head_dim]
        # sum agggregation
        out = super().aggregate(inputs, index, ptr, dim_size)
        return out.view(-1, self.node_dim)  # concat heads

In [ ]:
class Temporal_GNN(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim

        # used for merging semantic embeddings 
        self.init_embedding = nn.Linear(in_dim, hidden_dim)



    def forward(self):
        # embeddings

        # transfomer blocks

        # update lstm hidden state

        # scoring and link predicitons

        # loss and backprop
        pass

    def loss(self):
        pass

    def gen(self):
        pass